In [ ]:

#Order: setup → dataset → ALLaM-7B (baseline) → Jais-2-8B → Qwen3-8B → gemnini → comparison table.


In [ ]:
!pip install -q "transformers>=4.40.1" accelerate torch google-genai pandas bitsandbytes

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your-key-here"

JAIS Aunthentication


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Shared dataset: pragmatics items + ArPanEmo loader + fixed answer extraction

In [ ]:
import re, random, pandas as pd
random.seed(42)

SYSTEM_PROMPT = (
    "أنت خبير لغوي متخصص في تحليل النصوص العربية وفهم السياق والدلالات الضمنية "
    "في اللهجة السعودية. سأعرض عليك أسئلة اختيار من متعدد تتعلق بالفهم التداولي "
    "(المعنى المقصود خلف الكلام الظاهر).\n"
    "مهمتك:\n"
    "1. اقرأ السؤال والخيارات بعناية.\n"
    "2. حدد الإجابة الأصح بناءً على السياق والمعنى المقصود، وليس المعنى الحرفي فقط.\n"
    "3. أجب برقم الخيار الصحيح فقط (مثل: 1) دون أي شرح إضافي."
)
USER_TEMPLATE = "السؤال: {question}\nالخيارات:\n{options}\nالإجابة (رقم فقط):"

# --- Pragmatics set (placeholder — expand with your curated items) ---
TEST_ITEMS = [
    # --- sarcasm ---
    {"id": "sarcasm_01", "phenomenon": "sarcasm",
     "question": "بعد ما تأخر عنه صاحبه ساعتين، قال له: \"ما شاء الله عليك، دقيق مواعيد!\" -- وش يقصد؟",
     "options": ["1. يمدحه لأنه ملتزم بالمواعيد", "2. يعاتبه بشكل ساخر على تأخيره", "3. يسأله عن الساعة", "4. يشكره على الحضور"],
     "answer": 2},
    {"id": "sarcasm_02", "phenomenon": "sarcasm",
     "question": "بعد ما خرب السيارة وهو يتعلم يسوق، قال له أبوه: \"ما شاء الله عليك سواق محترف!\" -- وش يقصد؟",
     "options": ["1. يمدحه فعلاً لأنه سواق ماهر", "2. يعاتبه بسخرية على قيادته السيئة", "3. يشجعه يتعلم أكثر", "4. يسأله عن رخصة القيادة"],
     "answer": 2},
    {"id": "sarcasm_03", "phenomenon": "sarcasm",
     "question": "طلب منها تجهز الأكل بسرعة، وبعد ٣ ساعات قالت: \"تفضل، جاهز على السريع!\" فقال: \"ما شاء الله عليك، فعلاً سريعة!\" -- وش يقصد؟",
     "options": ["1. يمدح سرعتها", "2. يسخر من تأخرها الطويل", "3. يشكرها على الأكل", "4. يسألها عن الوقت"],
     "answer": 2},
    {"id": "sarcasm_04", "phenomenon": "sarcasm",
     "question": "بعد ما نسي صديقه ميعاد عيد ميلاده، قال له: \"تسلم إيدك، ما نسيت ولا شي!\" -- وش يقصد؟",
     "options": ["1. يشكره لأنه تذكر", "2. يعاتبه بسخرية لأنه نسي", "3. يطلب منه هدية", "4. يهنئه بعيد ميلاده"],
     "answer": 2},
    {"id": "sarcasm_05", "phenomenon": "sarcasm",
     "question": "طالب ما ذاكر للاختبار وطلعت درجته متدنية، قال له زميله: \"الله يعطيك العافية، تعبت من كثر المذاكرة!\" -- وش يقصد؟",
     "options": ["1. يمدح مجهوده بالمذاكرة", "2. يسخر من كسله وعدم مذاكرته", "3. يسأله عن نتيجة الاختبار", "4. يطلب يذاكر معه"],
     "answer": 2},
    {"id": "sarcasm_06", "phenomenon": "sarcasm",
     "question": "بعد ما وصل الاجتماع متأخر نص ساعة، قالت له المديرة: \"يعطيك العافية على الالتزام بالوقت!\" -- وش تقصد؟",
     "options": ["1. تمدحه على التزامه", "2. تعاتبه بسخرية على تأخيره", "3. تسأله عن سبب الغياب", "4. تشكره على الحضور"],
     "answer": 2},

    # --- indirect speech acts ---
    {"id": "indirect_01", "phenomenon": "indirect_speech_act",
     "question": "قالت الأم لابنها وهو جالس قدام التلفزيون: \"ما عندك واجبات اليوم؟\" -- وش تقصد؟",
     "options": ["1. تسأله سؤال عادي عن وجود واجبات", "2. تطلب منه بشكل غير مباشر يروح يذاكر", "3. تخبره إنه ما عليه واجبات", "4. تطلب منه يطفي التلفزيون بس"],
     "answer": 2},
    {"id": "indirect_02", "phenomenon": "indirect_speech_act",
     "question": "قال الزوج لزوجته والصالة حارة جدًا: \"ودي أعرف ليش النافذة مفتوحة؟\" -- وش يقصد؟",
     "options": ["1. يسأل سؤال عادي عن سبب فتح النافذة", "2. يطلب منها بشكل غير مباشر تسكر النافذة أو تشغل المكيف", "3. يشتكي من الحر بدون أي طلب", "4. يطلب منها تفتح نافذة ثانية"],
     "answer": 2},
    {"id": "indirect_03", "phenomenon": "indirect_speech_act",
     "question": "قالت المعلمة للطالب اللي يتكلم بالفصل: \"ما أدري ليش ما أقدر أكمل الشرح؟\" -- وش تقصد؟",
     "options": ["1. تسأل فعلاً عن سبب توقفها", "2. تطلب منه بشكل غير مباشر يسكت", "3. تشتكي من صعوبة الدرس", "4. تطلب منه يشرح بدالها"],
     "answer": 2},
    {"id": "indirect_04", "phenomenon": "indirect_speech_act",
     "question": "قال الضيف لصاحب البيت: \"الجو برد شوي هنا مو؟\" -- وش يقصد؟",
     "options": ["1. يسأل عن درجة الحرارة بالخارج", "2. يطلب بشكل غير مباشر يسكرون المكيف أو يعطونه غطا", "3. يشتكي من الطقس العام", "4. يطلب يطلع يمشي"],
     "answer": 2},
    {"id": "indirect_05", "phenomenon": "indirect_speech_act",
     "question": "قالت الأم وهي تشوف طبق الحلا فاضي: \"مين قاعد ياكل قبل الغدا؟\" -- وش تقصد؟",
     "options": ["1. تسأل سؤال فعلي عشان تعرف مين اللي أكل", "2. تعاتب بشكل غير مباشر اللي أكل الحلا قبل الوقت", "3. توزع الحلا على الكل", "4. تطلب حد يطبخ حلا جديد"],
     "answer": 2},
    {"id": "indirect_06", "phenomenon": "indirect_speech_act",
     "question": "قال المدير للموظف اللي طلع متأخر: \"الساعة كم عندك؟\" -- وش يقصد؟",
     "options": ["1. يسأل فعلاً عن الوقت عشان ساعته وقفت", "2. يعاتبه بشكل غير مباشر على تأخيره عن الدوام", "3. يطلب منه يشتري له ساعة", "4. يسأل عن مواعيد الاجتماع القادم"],
     "answer": 2},

    # --- idiom / proverb ---
    {"id": "idiom_01", "phenomenon": "idiom_proverb",
     "question": "قال له: \"اللي ما يعرف الصقر يشويه\" -- وش يقصد بهالمثل؟",
     "options": ["1. نصيحة بطريقة طبخ الصقر", "2. من يجهل قيمة الشيء الثمين يهدره أو يسيء التعامل معه", "3. وصف لنوع معين من الطيور", "4. تحذير من أكل الصقور"],
     "answer": 2},
    {"id": "idiom_02", "phenomenon": "idiom_proverb",
     "question": "قال المعلم للطالب: \"اللي فات مات\" -- وش يقصد بهالمثل؟",
     "options": ["1. نصيحة بنسيان الماضي والتركيز على المستقبل", "2. الحديث عن شخص متوفي فعلاً", "3. وصف لحدث تاريخي", "4. تحذير من الموت"],
     "answer": 1},
    {"id": "idiom_03", "phenomenon": "idiom_proverb",
     "question": "قالت الجدة: \"يد وحدة ما تصفق\" -- وش تقصد؟",
     "options": ["1. وصف حرفي لليد أثناء التصفيق", "2. التعاون والعمل الجماعي ضروري لتحقيق الأمور", "3. نصيحة بعدم التصفيق بيد وحدة", "4. وصف لحركة رياضية معينة"],
     "answer": 2},
    {"id": "idiom_04", "phenomenon": "idiom_proverb",
     "question": "قال الأب لابنه: \"اللي يزرع يحصد\" -- وش يقصد؟",
     "options": ["1. نصيحة زراعية عن كيفية الزراعة", "2. من يجتهد ويعمل بيحصل على نتيجة جهوده لاحقًا", "3. وصف لموسم الحصاد", "4. تحذير من ترك الأرض بدون زراعة"],
     "answer": 2},
    {"id": "idiom_05", "phenomenon": "idiom_proverb",
     "question": "قالت الأم: \"اللي ما له أول ما له تالي\" -- وش تقصد؟",
     "options": ["1. وصف لترتيب الأشياء", "2. من لا يهتم بأصله وجذوره ما راح يوصل لمستقبل جيد", "3. نصيحة بالبداية من الآخر", "4. وصف لقصة معينة"],
     "answer": 2},
    {"id": "idiom_06", "phenomenon": "idiom_proverb",
     "question": "قال الجد: \"اللي يشوف بلوى غيره تهون عليه بلواه\" -- وش يقصد؟",
     "options": ["1. وصف حرفي لمشاهدة مشاكل الناس", "2. رؤية معاناة الآخرين تجعل الشخص يتقبل مشكلته بشكل أفضل", "3. نصيحة بعدم الاهتمام بمشاكل الآخرين", "4. تحذير من التدخل بمشاكل غيرك"],
     "answer": 2},
    # --- add more here to reach 15-20 per phenomenon for a stronger sample ---
]

# --- ArPanEmo emotion-classification set (adjust path to where your CSV lives) ---
ARPANEMO_PATH = "/content/drive/MyDrive/Colab Notebooks/ArPanEmo_test.csv"
N_PER_LABEL = 5

def load_arpanemo(path, n_per_label=5):
    df = pd.read_csv(path)
    cols = {c.lower().strip(): c for c in df.columns}
    df = df.rename(columns={cols["post"]: "Post", cols["label"]: "Label"})
    df = df[df["Post"].astype(str).str.strip() != "..."]
    df = df[df["Post"].astype(str).str.len() > 3]
    labels = sorted(df["Label"].astype(str).str.strip().unique())
    print(f"ArPanEmo labels found: {labels}  ({len(df)} usable rows after filtering)")

    items = []
    for label in labels:
        subset = df[df["Label"].astype(str).str.strip() == label]
        sample = subset.sample(n=min(n_per_label, len(subset)), random_state=42)
        for i, row in sample.iterrows():
            post = str(row["Post"]).strip()
            shuffled = labels.copy()
            random.shuffle(shuffled)
            options = [f"{idx+1}. {lbl}" for idx, lbl in enumerate(shuffled)]
            items.append({
                "id": f"emotion_{i}", "phenomenon": "emotion_classification",
                "question": f'ما هو الشعور الأساسي المعبر عنه في هذا المنشور؟\n\nالمنشور: "{post}"',
                "options": options, "answer": shuffled.index(label) + 1,
            })
    return items

EMOTION_ITEMS = load_arpanemo(ARPANEMO_PATH, N_PER_LABEL)

ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def extract_choice(raw_text, n_options):
    """Return predicted option number (1..n_options), or None if unparseable.
    Matches 1-2 digit numbers — required for tasks with 10+ options (ArPanEmo),
    where a single-digit-only regex truncates '10'/'11' down to '1'."""
    text = raw_text.translate(ARABIC_DIGITS)
    match = re.search(r"\d{1,2}", text)
    if match:
        val = int(match.group())
        if 1 <= val <= n_options:
            return val
    return None

def build_messages(item):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(
            question=item["question"], options="\n".join(item["options"]))},
    ]

print(f"Pragmatics items: {len(TEST_ITEMS)}  |  Emotion items: {len(EMOTION_ITEMS)}")

Helpers (model runner + scoring)

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

def score_all(items, generate_fn):
    rows = []
    for item in items:
        raw = generate_fn(item)
        pred = extract_choice(raw, len(item["options"]))
        rows.append({"id": item["id"], "phenomenon": item["phenomenon"],
                      "gold": item["answer"], "pred": pred,
                      "correct": pred == item["answer"], "raw_output": raw.strip()})
    return rows

import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def run_hf_model(repo_id, items, max_new_tokens=8, enable_thinking=False):
    tokenizer = AutoTokenizer.from_pretrained(repo_id)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        repo_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()

    def generate(item):
        chat_kwargs = {"tokenize": False, "add_generation_prompt": True}
        try:
            prompt = tokenizer.apply_chat_template(build_messages(item), enable_thinking=enable_thinking, **chat_kwargs)
        except TypeError:
            prompt = tokenizer.apply_chat_template(build_messages(item), **chat_kwargs)
        inputs = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
        raw = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        if "</think>" in raw:
            raw = raw.split("</think>")[-1]
        return raw

    rows = score_all(items, generate)
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    return rows


def report(name, rows):
    acc = sum(r["correct"] for r in rows) / len(rows)
    print(f"{name}: accuracy = {acc:.1%}  ({sum(r['correct'] for r in rows)}/{len(rows)})")
    for r in rows:
        mark = "✓" if r["correct"] else "✗"
        print(f"  [{r['id']:15s}] gold={r['gold']} pred={r['pred']}  {mark}  raw={r['raw_output'][:30]!r}")

all_results = {}  # "<model>__pragmatics" / "<model>__emotion" -> rows
MODEL_CONFIGS = [
    {"name": "ALLaM-7B-Instruct-preview", "repo_id": "ALLaM-AI/ALLaM-7B-Instruct-preview", "max_new_tokens": 8},
    {"name": "Jais-2-8B-Chat",            "repo_id": "inceptionai/Jais-2-8B-Chat",         "max_new_tokens": 8},
    {"name": "Qwen3-8B",                  "repo_id": "Qwen/Qwen3-8B",                       "max_new_tokens": 200},
]

In [ ]:
!pip install -q groq
import time
from groq import Groq

client_groq = Groq()  # reads GROQ_API_KEY from environment

def generate_groq(item, model_name="openai/gpt-oss-120b", max_retries=4):
    for attempt in range(max_retries):
        try:
            resp = client_groq.chat.completions.create(
                model=model_name,
                messages=build_messages(item),
                max_tokens=300,              # was 12 — reasoning tokens need room
                temperature=0,
                reasoning_effort="low",       # if Groq rejects this kwarg, just remove the line
            )
            raw = resp.choices[0].message.content or ""
            if "</think>" in raw:            # strip any leaked reasoning block, same as Qwen3 fix
                raw = raw.split("</think>")[-1]
            return raw
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  retry {attempt+1}/{max_retries} after error: {e} — waiting {wait}s")
            time.sleep(wait)

In [ ]:
rows = score_all(TEST_ITEMS, lambda item: generate_groq(item))
all_results["GPT-OSS-120B-Groq__pragmatics"] = rows
report("GPT-OSS-120B-Groq", rows)

rows = score_all(EMOTION_ITEMS, lambda item: generate_groq(item))
all_results["GPT-OSS-120B-Groq__emotion"] = rows
report("GPT-OSS-120B-Groq", rows)

Run every model on both tasks (one pass)

In [ ]:
for cfg in MODEL_CONFIGS:
    print(f"\n=== {cfg['name']} — pragmatics ===")
    rows = run_hf_model(cfg["repo_id"], TEST_ITEMS, max_new_tokens=cfg["max_new_tokens"])
    all_results[f"{cfg['name']}__pragmatics"] = rows
    report(cfg["name"], rows)

    print(f"\n=== {cfg['name']} — emotion (ArPanEmo) ===")
    rows = run_hf_model(cfg["repo_id"], EMOTION_ITEMS, max_new_tokens=max(cfg["max_new_tokens"], 12))
    all_results[f"{cfg['name']}__emotion"] = rows
    report(cfg["name"], rows)

Comparison tables (kept separate — different chance-accuracy: 25% for 4-option pragmatics vs ~9% for 11-option emotion)

In [ ]:
import pandas as pd, json

def build_table(suffix):
    summary = []
    for key, rows in all_results.items():
        if not key.endswith(suffix):
            continue
        model_name = key[: -len(suffix)]
        overall = sum(r["correct"] for r in rows) / len(rows)
        row = {"model": model_name, "overall_accuracy": round(overall, 3)}
        for ph in sorted(set(r["phenomenon"] for r in rows)):
            ph_rows = [r for r in rows if r["phenomenon"] == ph]
            row[ph] = round(sum(r["correct"] for r in ph_rows) / len(ph_rows), 3)
        summary.append(row)
    return pd.DataFrame(summary).set_index("model")

print("=== Pragmatics (sarcasm / indirect / idiom) ===")
df_prag = build_table("__pragmatics")
display(df_prag)

print("\n=== Emotion classification (ArPanEmo, 11-way) ===")
df_emo = build_table("__emotion")
display(df_emo)

df_prag.to_csv("comparison_pragmatics.csv")
df_emo.to_csv("comparison_emotion.csv")
with open("comparison_results_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print("\nSaved comparison_pragmatics.csv, comparison_emotion.csv, comparison_results_raw.json")

Inspect Qwen3's emotion misclassifications

In [ ]:
# Look up original label text (not just index) for Qwen3's wrong emotion answers
def label_for(item_id, idx, item_list):
    if idx is None:
        return None
    item = next(i for i in item_list if i["id"] == item_id)
    return item["options"][idx - 1].split(". ", 1)[1]

qwen_emotion_rows = all_results["Qwen3-8B__emotion"]
wrong = [r for r in qwen_emotion_rows if not r["correct"]]
print(f"{len(wrong)}/{len(qwen_emotion_rows)} wrong\n")

for r in wrong:
    gold_label = label_for(r["id"], r["gold"], EMOTION_ITEMS)
    pred_label = label_for(r["id"], r["pred"], EMOTION_ITEMS)
    print(f"[{r['id']}] gold={gold_label!r}  pred={pred_label!r}  raw={r['raw_output'][:20]!r}")

In [ ]:
VALENCE = {
    "anger": "negative", "disgust": "negative", "fear": "negative",
    "sadness": "negative", "pessimism": "negative",
    "happiness": "positive", "optimism": "positive",
    "neutral": "neutral", "confusion": "neutral",
    "anticipation": "positive", "surprise": "neutral",
}

def label_for(item_id, idx, item_list):
    if idx is None: return None
    item = next(i for i in item_list if i["id"] == item_id)
    return item["options"][idx - 1].split(". ", 1)[1]

rows = all_results["Qwen3-8B__emotion"]
same_valence_errors = 0
diff_valence_errors = 0
total_errors = 0

for r in rows:
    if r["correct"]:
        continue
    gold = label_for(r["id"], r["gold"], EMOTION_ITEMS)
    pred = label_for(r["id"], r["pred"], EMOTION_ITEMS)
    if pred is None:
        continue
    total_errors += 1
    if VALENCE.get(gold) == VALENCE.get(pred):
        same_valence_errors += 1
    else:
        diff_valence_errors += 1

print(f"Total scored errors: {total_errors}")
print(f"Same-valence confusions (right polarity, wrong specific emotion): {same_valence_errors} ({same_valence_errors/total_errors:.1%})")
print(f"Cross-valence confusions (wrong polarity entirely): {diff_valence_errors} ({diff_valence_errors/total_errors:.1%})")